# Pixel's Tiny Spam Filter
**Pixel's AI Field Guide: Missions for Curious Humans** - Spam-filter notebook, Version 1.0, 2026-11-06

**[MISSION BRIEF]** Dr. Ada Okafor's inbox at the Curiosity Lab is full of junk. Pixel, the very literal lab robot, wants to help. Nobody will write the rules for Pixel: Pixel must **learn** them from 12 example emails that a human has already labelled "spam" or "not spam".

All emails in this notebook are **made-up examples** written for this activity. The numbers you will see are illustrative, not measurements of any real spam filter.

**You do not need to know Python.** You only run the cells in order and change the text inside quotation marks in the EXPERIMENT ZONE.

## How to open and run this notebook

**Option A - Google Colab (online, free, needs a Google account):**
1. Download `tiny_spam_filter.ipynb` to your computer.
2. Go to https://colab.research.google.com and sign in with a Google account.
3. Choose **File -> Upload notebook** and pick the downloaded file. Colab opens it and keeps your personal copy in your Google Drive, so your changes do not affect anyone else.
4. Run each cell from top to bottom: click inside a cell and press **Shift+Enter** (or click the round play button on its left). Or use **Runtime -> Run all**.

**Option B - Jupyter on your own computer (offline):** open the file in Jupyter Notebook or JupyterLab and run the cells in order with **Shift+Enter**.

**Option C - no notebook at all:** run `python3 tiny_spam_filter.py` (the same code in one file) and compare with `expected-output.txt`.

If you change something in the EXPERIMENT ZONE, re-run **all** cells from that cell downwards (Colab: **Runtime -> Run after**), otherwise Pixel keeps using the old data.

## Step 0 - Load the one tool we need
`re` is Python's built-in text-searching tool. We use it to cut sentences into words. Nothing needs installing.

In [ ]:
import re

## Step 1 - The training data (EXPERIMENT ZONE)
Each line is one example: `("email text", "label")`. The label is the right answer, given by a human. This is **supervised learning**: learning from labelled examples.

The **test emails** are kept separate. Pixel never trains on them; we use them only to check how well Pixel learned, like a quiz with new questions.

**[PREDICT]** Read the training emails. Write down three words you think Pixel will find "spammy" and three it will find "safe".

In [ ]:
# ======================= EXPERIMENT ZONE (change me!) =======================
# Each training example is (email text, label). Labels: "spam" or "not spam".
# Try: add an email, delete one, or swap a label, then run again and compare.
TRAINING_DATA = [
    ("WIN a FREE prize now! Click here", "spam"),
    ("Cheap pills, buy now, limited offer", "spam"),
    ("You are a winner! Claim your cash prize today", "spam"),
    ("Free money!!! Click the link to claim", "spam"),
    ("Limited time offer: win a free phone", "spam"),
    ("Urgent: your account wins cash, click now", "spam"),
    ("Lab meeting moved to Tuesday at 10", "not spam"),
    ("Can you send me the robot battery report?", "not spam"),
    ("Lunch today? The cafe has soup", "not spam"),
    ("Dr. Okafor shared the meeting notes", "not spam"),
    ("Reminder: return the library book on Friday", "not spam"),
    ("Thanks for fixing the robot wheel today", "not spam"),
]

# Test emails Pixel has NEVER seen. The label is the right answer, used
# only to CHECK Pixel afterwards (Pixel does not get to peek).
TEST_DATA = [
    ("Claim your free prize now", "spam"),
    ("Click here for cheap cash", "spam"),
    ("Buy a cheap phone now, limited offer", "spam"),
    ("Congratulations, you have been selected", "spam"),
    ("Meeting notes for the robot lab", "not spam"),
    ("Please send the battery report by Friday", "not spam"),
    ("Are you free for lunch today?", "not spam"),
    ("You won a prize in the lab quiz", "not spam"),
]

# Type your own message here and see how Pixel scores every word.
MY_MESSAGE = "Free lunch in the lab today, click here to sign up"
# ============================================================================

## Step 2 - How Pixel learns (the recipe)
- `words_in` turns an email into a set of lowercase words (`"WIN a FREE prize!"` -> `win, a, free, prize`).
- `train` counts how many spam and how many not-spam emails contain each word, then gives each word a **score**:
  - above 1 = appears more often in spam (spammy),
  - below 1 = appears more often in not-spam (safe),
  - exactly 1 = no evidence either way.
- `classify` **multiplies** the scores of all the words in a new email. A final score above 1 means "spam".

Technically speaking, this is a simplified version of a **naive Bayes classifier**. It is "naive" because it treats every word separately and ignores word order and meaning.

In [ ]:
def words_in(text):
    """Split text into lowercase words, e.g. 'WIN a FREE prize!' ->
    {'win', 'a', 'free', 'prize'}. A set ignores repeats in one email."""
    return set(re.findall(r"[a-z]+", text.lower()))


def train(examples):
    """TRAINING: count, for every word, how many spam and how many
    not-spam emails contain it, then turn counts into word scores."""
    n_spam = sum(1 for _, label in examples if label == "spam")
    n_ham = len(examples) - n_spam                 # "ham" = not spam
    spam_count, ham_count = {}, {}
    for text, label in examples:
        counts = spam_count if label == "spam" else ham_count
        for w in words_in(text):
            counts[w] = counts.get(w, 0) + 1
    scores = {}
    for w in set(spam_count) | set(ham_count):
        # Share of spam emails containing w, divided by the share of
        # not-spam emails containing w. The "+ 1" and "+ 2" (called
        # smoothing) stop a word seen only once from getting a score of 0
        # or "infinity", which would overrule every other word.
        in_spam = (spam_count.get(w, 0) + 1) / (n_spam + 2)
        in_ham = (ham_count.get(w, 0) + 1) / (n_ham + 2)
        scores[w] = in_spam / in_ham
    return scores, spam_count, ham_count


def classify(text, scores):
    """INFERENCE: multiply the scores of the words in the email.
    Unknown words get score 1, so they change nothing."""
    total = 1.0
    for w in sorted(words_in(text)):
        total *= scores.get(w, 1.0)
    total = round(total, 6)   # remove tiny computer rounding errors (1.0000000002)
    # A score of exactly 1 means "no evidence": Pixel then says NOT spam.
    return ("spam" if total > 1.0 else "not spam"), total

## Step 3 - Train Pixel and look inside its "brain"
**[PREDICT]** Will the word `a` be spammy, safe, or neutral? (Check the training emails.)

In [ ]:
# ---- 1. Training ------------------------------------------------------------
scores, spam_count, ham_count = train(TRAINING_DATA)
print("Trained on", len(TRAINING_DATA), "made-up emails;", len(scores), "different words.\n")

# ---- 2. Show what Pixel learned (most spammy and most safe words) -----------
ranked = sorted(scores, key=lambda w: (-scores[w], w))
print("Most SPAMMY words   (score | in spam | in not-spam)")
for w in ranked[:6]:
    print(f"  {w:<10} {scores[w]:5.2f} | {spam_count.get(w, 0)} | {ham_count.get(w, 0)}")
print("Most SAFE words")
for w in sorted(scores, key=lambda w: (scores[w], w))[:6]:
    print(f"  {w:<10} {scores[w]:5.2f} | {spam_count.get(w, 0)} | {ham_count.get(w, 0)}")

**[OBSERVE]** Did any surprising word show up in the spammy list? Pixel learned that the tiny word `a` is spammy, because by coincidence it appears in 3 spam emails and in no normal ones. With only 12 examples, accidents in the data become "rules". This is called **overgeneralizing**.

## Step 4 - Test Pixel on 8 emails it has never seen
**[PREDICT]** For each test email, guess what Pixel will say **before** running the cell. Which ones might fool Pixel?

In [ ]:
# ---- 3. Test on emails Pixel has never seen ---------------------------------
print("\nTEST RESULTS (final score above 1 means spam)")
correct = 0
for text, true_label in TEST_DATA:
    guess, total = classify(text, scores)
    ok = guess == true_label
    correct += ok
    mark = "correct" if ok else "WRONG  "
    print(f"  {mark} score {total:9.3f} -> {guess:<8} | {text}")
print(f"\nAccuracy: {correct} of {len(TEST_DATA)} correct "
      f"({100 * correct / len(TEST_DATA):.1f}%)")

**[OBSERVE]** Expected result: **5 of 8 correct (62.5%)**. Look at the three mistakes:
1. *"Congratulations, you have been selected"* - almost every word is **new** to Pixel (score 1 = no evidence), so it defaults to "not spam".
2. *"Are you free for lunch today?"* - Pixel only knows `free` from spam (`free money`). It cannot tell that "free for lunch" means something different.
3. *"You won a prize in the lab quiz"* - `prize` and `a` look spammy. Pixel does not know that `won` is related to `win`, because to Pixel they are just different strings of letters.

## Step 5 - Try your own message
Change `MY_MESSAGE` in the EXPERIMENT ZONE (Step 1), then re-run all cells from Step 1 down. Pixel shows the score of every word, so you can see **why** it decided.

In [ ]:
# ---- 4. Your own message, word by word --------------------------------------
guess, total = classify(MY_MESSAGE, scores)
print(f'\nYOUR MESSAGE: "{MY_MESSAGE}"')
for w in sorted(words_in(MY_MESSAGE)):
    print(f"  {w:<10} {scores.get(w, 1.0):5.2f}" + ("" if w in scores else "  (never seen)"))
print(f"  Final score {total:.3f} -> Pixel says: {guess.upper()}")

## Experiments
**[TRY IT] Experiment A - better training data.** In Step 1, add these two lines inside `TRAINING_DATA` (just before the closing `]`), then re-run everything:
```
    ("Congratulations! You have been selected to win cash", "spam"),
    ("Are you free for a chat on Friday?", "not spam"),
```
**[PREDICT]** first: how many test emails will be correct now?

**[TRY IT] Experiment B - your own spam.** Write a message that you think Pixel will get wrong. Can you fool it with words it has never seen, or by mixing safe and spammy words?

**[TRY IT] Experiment C - change one label.** Change the label of one training email from `"not spam"` to `"spam"` (a human labelling mistake). Which word scores change?

<details><summary>[DEBRIEF] Expected results (open after you have tried)</summary>

- **Experiment A:** accuracy rises to **8 of 8 (100%)**. Two new examples taught Pixel that `congratulations`, `selected` and `win` are spammy and that `free` and `for` also appear in normal emails. But look closely: *"You won a prize in the lab quiz"* now scores **exactly 1.000**, a tie that Pixel only gets "right" because ties count as not spam. And we chose the new examples **after looking at the test mistakes**. That is a kind of cheating: the test set no longer measures performance on truly unseen email. Real projects keep a separate, untouched test set for this reason.
- **Experiment B:** messages made of brand-new words always score 1 (not spam). Messages that mix many safe words (`meeting`, `robot`, `the`) with one spammy word are usually classed as not spam.
- **Experiment C:** every word in the relabelled email moves toward "spammy". Bad labels produce bad learning: "garbage in, garbage out".
</details>

## [EXPLAIN] Reflection
1. Pixel was never given a rule like "the word *free* means spam". Where did that idea come from?
2. Why is a test set of emails Pixel has never seen more honest than re-testing it on the training emails?
3. Name two things a human understands about *"Are you free for lunch?"* that Pixel's word scores cannot capture.

**[LAB NOTE] Limitations of this model.** Real spam filters learn from millions of emails and use many more clues (sender, links, word order, attachments, user reports). This toy uses 12 made-up emails, ignores word order and meaning, and gives every rare word the same weight. Its accuracy figures say nothing about real-world spam filters.